# Medical Insurance Charges — Exploratory Data Analysis & Prediction

**Goal:** explore the medical insurance dataset and build a baseline model to estimate insurance charges from demographic and lifestyle features.

> This is an educational regression project. Predictions are estimates, not medical or financial advice. The dataset reflects its own sample and should not be treated as representative of every population.

## Project workflow
1. Load and validate the data
2. Explore distributions and relationships
3. Clean duplicates and check missing values
4. Split data before fitting preprocessing steps
5. Train baseline regression models
6. Evaluate performance on held-out test data
7. Summarize limitations and next steps

**Dataset required:** `insurance.csv` in the same folder as this notebook. Expected columns: `age`, `sex`, `bmi`, `children`, `smoker`, `region`, and `charges`.

In [ ]:
# If needed, install dependencies in your environment:
# %pip install pandas numpy matplotlib seaborn scikit-learn

from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

In [ ]:
data_path = Path("insurance.csv")
if not data_path.exists():
    raise FileNotFoundError(
        "Could not find insurance.csv. Place the dataset in the same folder as this notebook "
        "or update data_path to the correct location."
    )

df = pd.read_csv(data_path)
df.head()

## 1. Understand the data

The target variable is `charges`. The remaining columns are candidate predictors.

In [ ]:
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
display(df.head())
df.info()

In [ ]:
# Validate expected columns before continuing
expected_columns = {"age", "sex", "bmi", "children", "smoker", "region", "charges"}
missing_columns = expected_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing expected columns: {sorted(missing_columns)}")

display(df.describe(include="all").T)

In [ ]:
print("Missing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

print(f"Duplicate rows: {df.duplicated().sum()}")
print("\nCategory values:")
for col in ["sex", "smoker", "region"]:
    print(f"{col}: {sorted(df[col].dropna().astype(str).unique())}")

## 2. Exploratory data analysis (EDA)

Plots are exploratory. Patterns in this dataset do not prove that a feature causes higher or lower charges.

In [ ]:
numeric_cols = ["age", "bmi", "children", "charges"]
df[numeric_cols].hist(figsize=(12, 8), bins=25, edgecolor="white")
plt.suptitle("Distributions of numeric variables")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["sex", "smoker", "region"]):
    sns.countplot(data=df, x=col, ax=ax)
    ax.set_title(f"Count by {col}")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="bmi", y="charges", hue="smoker", alpha=0.65)
plt.title("Insurance charges vs. BMI, by smoking status")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="smoker", y="charges")
plt.title("Insurance charges by smoking status")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Correlation among numeric variables")
plt.tight_layout()
plt.show()

## 3. Data quality and preparation

Remove exact duplicate records and validate numeric ranges. Do not convert the whole DataFrame to integers: `bmi` and `charges` contain meaningful decimal values.

For a realistic evaluation, split the data into training and test sets **before** fitting imputers, encoders, or scalers. The scikit-learn pipeline below learns preprocessing only from the training data.

In [ ]:
df = df.drop_duplicates().copy()

# Convert numeric fields safely; invalid values become missing and are imputed inside the pipeline.
for col in ["age", "bmi", "children", "charges"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Rows without the target cannot be used for supervised training.
df = df.dropna(subset=["charges"]).copy()

# Basic plausibility checks (flag only; do not silently remove records)
display(pd.DataFrame({
    "check": ["age <= 0", "bmi <= 0", "children < 0", "charges < 0"],
    "count": [
        (df["age"] <= 0).sum(),
        (df["bmi"] <= 0).sum(),
        (df["children"] < 0).sum(),
        (df["charges"] < 0).sum()
    ]
}))
print(f"Rows after removing duplicates and missing target: {len(df):,}")

## 4. Train/test split and preprocessing

The target (`charges`) is kept separate from the predictors. `bmi_category` is not included in the model because it is derived from BMI and duplicates information already present in the raw BMI feature.

- Numeric features: median imputation + standard scaling
- Categorical features: most-frequent imputation + one-hot encoding
- Test data remains unseen while preprocessing is fitted

In [ ]:
target = "charges"
X = df.drop(columns=[target])
y = df[target]

numeric_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print(f"Training rows: {X_train.shape[0]:,}")
print(f"Test rows: {X_test.shape[0]:,}")

## 5. Train baseline models

A mean-prediction baseline provides a useful reference. Linear regression is a simple interpretable model; random forest can capture nonlinear relationships.

In [ ]:
models = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Linear regression": LinearRegression(),
    "Ridge regression": Ridge(alpha=1.0),
    "Random forest": RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

fitted_models = {}
results = []

for name, estimator in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    pipe.fit(X_train, y_train)
    predictions = pipe.predict(X_test)

    fitted_models[name] = pipe
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions)
    })

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(results_df.round(2))

## 6. Evaluate the best model on the held-out test set

The lowest test RMSE is selected for this demonstration. Compare MAE and RMSE as well as R²; no single metric tells the whole story.

In [ ]:
best_model_name = results_df.loc[0, "Model"]
best_model = fitted_models[best_model_name]
test_predictions = best_model.predict(X_test)

print(f"Selected model: {best_model_name}")
print(f"MAE:  {mean_absolute_error(y_test, test_predictions):,.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, test_predictions)):,.2f}")
print(f"R²:   {r2_score(y_test, test_predictions):.3f}")

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=test_predictions, alpha=0.65)
low = min(y_test.min(), test_predictions.min())
high = max(y_test.max(), test_predictions.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual charges")
plt.ylabel("Predicted charges")
plt.title(f"Actual vs. predicted charges — {best_model_name}")
plt.tight_layout()
plt.show()

In [ ]:
residuals = y_test - test_predictions
plt.figure(figsize=(7, 5))
sns.scatterplot(x=test_predictions, y=residuals, alpha=0.65)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted charges")
plt.ylabel("Residual (actual - predicted)")
plt.title("Residual analysis")
plt.tight_layout()
plt.show()

## 7. Conclusions and limitations

Complete this section after running the notebook and reviewing the actual metrics and plots.

Suggested points to discuss:
- Which model performed best on the test set, and by how much did it improve over the mean baseline?
- How large is the typical prediction error (MAE) in the dataset's currency units?
- Do residuals show patterns suggesting the model misses some groups or high-cost cases?
- The dataset may be limited in size, geography, time period, and available features.
- Associations are not causal explanations. Do not use this educational model to make real insurance decisions.

## Reproducibility checklist

- [ ] `insurance.csv` is present in the project folder.
- [ ] Restart the kernel and run all cells from top to bottom.
- [ ] Confirm there are no errors and review the metric outputs.
- [ ] Update this conclusion with results from your actual run.
- [ ] Add a clear README and avoid committing private or licensed data unless you have permission.